In [2]:
# Cell 1: Environment-Aware Installation
import torch

# 1. Install standard libraries
!pip install -q "numpy>=2.0.0" "node2vec" --no-deps
!pip install -q ogb torch-geometric igraph leidenalg

# 2. Dynamically detect PyTorch and CUDA versions
pt_version = torch.__version__.split('+')[0]
cuda = torch.version.cuda
cuda_suffix = f"cu{cuda.replace('.', '')}" if cuda else "cpu"
wheel_url = f"https://data.pyg.org/whl/torch-{pt_version}+{cuda_suffix}.html"

# 3. Install pyg-lib from the correct PyG wheel index
!pip install -q pyg-lib -f {wheel_url}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 883.6 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 34.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 72.5 MB/s eta 0:00:00:00:0100:01


In [3]:
# Cell 2: Imports & Master Configuration
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm

# Network Processing & Community Detection
import networkx as nx
import igraph as ig
import leidenalg

# PyTorch & Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch_geometric.nn import Node2Vec as PyG_Node2Vec
from ogb.linkproppred import PygLinkPropPredDataset, Evaluator

# --- GLOBAL CONFIGURATION ---
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Node2Vec Hyperparameters
P_PARAM = 1.0            
Q_PARAM = 1.0            
EMBEDDING_DIM = 128      
WALKS_PER_NODE = 10      
WALK_LENGTH = 40         
CONTEXT_SIZE = 20        
N2V_EPOCHS = 3           
N2V_LR = 0.01

# Leiden Community Hyperparameters
LEIDEN_RESOLUTION = 1.0  
LEIDEN_OBJECTIVE = 'Modularity' 

# Set reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Target Device: {DEVICE} | Environment successfully initialized.")

Target Device: cuda | Environment successfully initialized.


In [4]:
# Cell 3: Dataset Ingestion & NetworkX Build
import networkx as nx
import numpy as np

print("--- Loading OGB Dataset ---")
_original_load = torch.load
torch.load = lambda *args, **kwargs: _original_load(*args, **{**kwargs, 'weights_only': False})

dataset = PygLinkPropPredDataset(name='ogbl-collab')
data = dataset[0]
evaluator = Evaluator(name='ogbl-collab')
split_edge = dataset.get_edge_split()
train_edge = split_edge['train']
valid_edge = split_edge['valid']
test_edge = split_edge['test']

torch.load = _original_load 

print("--- Splitting Training Edges (Preventing Data Leakage) ---")
np.random.seed(SEED)
all_train_edges = train_edge['edge'].numpy()
all_train_weights = train_edge['weight'].numpy()

# 90% for Graph Structure, 10% strictly for MLP Supervision
num_total_train = all_train_edges.shape[0]
perm = np.random.permutation(num_total_train)
split_idx = int(num_total_train * 0.90)

mp_indices = perm[:split_idx]
sup_indices = perm[split_idx:]

mp_edges = all_train_edges[mp_indices]
mp_weights = all_train_weights[mp_indices]
sup_edges = all_train_edges[sup_indices] # Saved for Cell 6

print(f"Message Passing Edges (For Graph): {len(mp_indices):,}")
print(f"Supervision Edges (For TG-MLP):    {len(sup_indices):,}")

# Build NetworkX Training Graph STRICTLY on Message Passing edges
G_train = nx.Graph()
for (u, v), w in zip(mp_edges, mp_weights):
    if G_train.has_edge(u, v):
        G_train[u][v]['weight'] += float(w)
    else:
        G_train.add_edge(int(u), int(v), weight=float(w))

--- Loading OGB Dataset ---
Total Nodes: 235,868 | Training Edges: 1,179,052
NetworkX Training Graph built | Nodes: 235,868 | Edges: 967,632


In [5]:
# Cell 4: Leiden Community Detection
print("--- Extracting Macro-Structure: Leiden Communities ---")

iG = ig.Graph.from_networkx(G_train)

# ModularityVertexPartition calculates standard modularity without resolution_parameter
partition = leidenalg.find_partition(
    iG, 
    leidenalg.ModularityVertexPartition,
    seed=SEED
)

nx_node_ids = iG.vs["_nx_name"]
community_dict = {nx_node_ids[i]: comm_id for i, comm_id in enumerate(partition.membership)}
comm_counts = Counter(community_dict.values())

print("=" * 50)
print(f"Total Distinct Communities Found: {len(comm_counts):,}")
print(f"Largest Community Size:           {max(comm_counts.values()):,}")
print(f"Average Community Size:           {int(np.mean(list(comm_counts.values()))):,}")
print("=" * 50)

--- Extracting Macro-Structure: Leiden Communities ---
Total Distinct Communities Found: 833
Largest Community Size:           16,089
Average Community Size:           283


In [6]:
# Cell 5: Node2Vec Execution
print("--- Generating Latent Topology (Node2Vec) ---")

# Node2Vec MUST only walk on the 90% Message Passing edges
mp_edges_tensor = torch.tensor(mp_edges).t().contiguous()
edge_index = torch.cat([mp_edges_tensor, mp_edges_tensor[[1, 0]]], dim=1)

model = PyG_Node2Vec(
    edge_index, embedding_dim=EMBEDDING_DIM, walk_length=WALK_LENGTH,
    context_size=CONTEXT_SIZE, walks_per_node=WALKS_PER_NODE,
    p=P_PARAM, q=Q_PARAM, num_nodes=data.num_nodes, sparse=True
).to(DEVICE)

loader = model.loader(batch_size=256, shuffle=True, num_workers=2)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=N2V_LR)

model.train()
for epoch in range(1, N2V_EPOCHS + 1):
    total_loss = 0
    progress_bar = tqdm(loader, desc=f"N2V Epoch {epoch}/{N2V_EPOCHS}", leave=True)
    for pos_rw, neg_rw in progress_bar:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(DEVICE), neg_rw.to(DEVICE))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

torch.save(model.embedding.weight.data.cpu(), 'node2vec_embeddings.pt')
n2v_embeddings = model.embedding.weight.data.cpu().numpy()
print(f"Embeddings saved. Matrix Shape: {n2v_embeddings.shape}")

--- Generating Latent Topology (Node2Vec) ---


N2V Epoch 1/3:   0%|          | 0/922 [00:00<?, ?it/s]

N2V Epoch 2/3:   0%|          | 0/922 [00:00<?, ?it/s]

N2V Epoch 3/3:   0%|          | 0/922 [00:00<?, ?it/s]

Embeddings saved. Matrix Shape: (235868, 128)


In [7]:
# Cell 6: Extract Streamlined Heuristics & Prepare Scaled Tensors
from sklearn.preprocessing import StandardScaler

print("--- Extracting Streamlined Features for MLP ---")
deg_dict = {n: G_train.degree(n) for n in G_train.nodes()}

def extract_streamlined_features(edge_array, G, comm_dict, desc="Extracting"):
    features = np.zeros((len(edge_array), 4))
    for i in tqdm(range(len(edge_array)), desc=desc):
        u, v = int(edge_array[i, 0]), int(edge_array[i, 1])
        features[i, 3] = 1.0 if comm_dict.get(u, -1) == comm_dict.get(v, -2) else 0.0
        
        if u in G and v in G:
            nu, nv = set(G[u]), set(G[v])
            common_neighbors = nu & nv
            if common_neighbors:
                features[i, 0] = sum(1.0 / np.log(deg_dict[w]) if deg_dict[w] > 1 else 0 for w in common_neighbors)
                union_size = len(nu) + len(nv) - len(common_neighbors)
                features[i, 1] = len(common_neighbors) / union_size if union_size > 0 else 0
                features[i, 2] = len(common_neighbors)
    return features

def get_hadamard_features(edge_array, embeddings, batch_size=50000, desc="Hadamard"):
    features = []
    for i in tqdm(range(0, len(edge_array), batch_size), desc=desc):
        batch = edge_array[i:i + batch_size]
        features.append(embeddings[batch[:, 0]] * embeddings[batch[:, 1]])
    return np.vstack(features)

np.random.seed(SEED)
# The TG-MLP is trained strictly on the 10% hidden supervision edges!
pos_train_edges = sup_edges
num_train = pos_train_edges.shape[0]

neg_train_edges = np.column_stack((
    np.random.randint(0, data.num_nodes, size=num_train),
    np.random.randint(0, data.num_nodes, size=num_train)
))

pos_val_edges = valid_edge['edge'].numpy()
neg_val_edges = valid_edge['edge_neg'].numpy()

print("\nExtracting Training Matrices...")
H_train_pos = get_hadamard_features(pos_train_edges, n2v_embeddings, desc="Train Had (Pos)")
H_train_neg = get_hadamard_features(neg_train_edges, n2v_embeddings, desc="Train Had (Neg)")
S_train_pos = extract_streamlined_features(pos_train_edges, G_train, community_dict, "Train Str (Pos)")
S_train_neg = extract_streamlined_features(neg_train_edges, G_train, community_dict, "Train Str (Neg)")

print("\nExtracting Validation Matrices...")
H_val_pos = get_hadamard_features(pos_val_edges, n2v_embeddings, desc="Val Had (Pos)")
H_val_neg = get_hadamard_features(neg_val_edges, n2v_embeddings, desc="Val Had (Neg)")
S_val_pos = extract_streamlined_features(pos_val_edges, G_train, community_dict, "Val Str (Pos)")
S_val_neg = extract_streamlined_features(neg_val_edges, G_train, community_dict, "Val Str (Neg)")

# Scale heuristics strictly on training distribution
scaler = StandardScaler()
S_train_scaled = scaler.fit_transform(np.vstack((S_train_pos, S_train_neg)))
S_train_pos_scaled = S_train_scaled[:num_train]
S_train_neg_scaled = S_train_scaled[num_train:]

S_val_pos_scaled = scaler.transform(S_val_pos)
S_val_neg_scaled = scaler.transform(S_val_neg)

X_train_n2v = torch.FloatTensor(np.vstack((H_train_pos, H_train_neg)))
X_train_heur = torch.FloatTensor(np.vstack((S_train_pos_scaled, S_train_neg_scaled)))
y_train_tensor = torch.FloatTensor(np.hstack((np.ones(num_train), np.zeros(num_train))))

X_val_pos_n2v, X_val_pos_heur = torch.FloatTensor(H_val_pos), torch.FloatTensor(S_val_pos_scaled)
X_val_neg_n2v, X_val_neg_heur = torch.FloatTensor(H_val_neg), torch.FloatTensor(S_val_neg_scaled)
print("\nFeature processing complete. Tensors ready for TG-MLP.")

--- Extracting Streamlined Features for MLP ---

Extracting Training Matrices...


Train Had (Pos):   0%|          | 0/24 [00:00<?, ?it/s]

Train Had (Neg):   0%|          | 0/24 [00:00<?, ?it/s]

Train Str (Pos):   0%|          | 0/1179052 [00:00<?, ?it/s]

Train Str (Neg):   0%|          | 0/1179052 [00:00<?, ?it/s]

...

Val Had (Pos):   0%|          | 0/2 [00:00<?, ?it/s]

Val Had (Neg):   0%|          | 0/2 [00:00<?, ?it/s]

Val Str (Pos):   0%|          | 0/60084 [00:00<?, ?it/s]

Val Str (Neg):   0%|          | 0/100000 [00:00<?, ?it/s]


Feature processing complete. Tensors ready for TG-MLP.


In [ ]:
# Cell 7: Topology-Gated Hybrid MLP (TG-MLP) Training
print("--- Initializing Topology-Gated PyTorch MLP ---")

train_dataset = TensorDataset(X_train_n2v, X_train_heur, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

class TopologyGatedMLP(nn.Module):
    def __init__(self, n2v_dim=128, heur_dim=4, hidden_dim=64):
        super(TopologyGatedMLP, self).__init__()
        
        self.n2v_proj = nn.Sequential(
            nn.Linear(n2v_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.heur_proj = nn.Sequential(
            nn.Linear(heur_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        self.gate_layer = nn.Linear(hidden_dim * 2, hidden_dim)
        
        # Classifier expanded to handle concatenated input (128D)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x_n2v, x_heur):
        h_n2v = self.n2v_proj(x_n2v)
        h_heur = self.heur_proj(x_heur)
        
        gate = torch.sigmoid(self.gate_layer(torch.cat([h_n2v, h_heur], dim=1)))
        gated_n2v = gate * h_n2v 
        
        # Concatenate instead of adding to force independent weighting
        fused = torch.cat([gated_n2v, h_heur], dim=1) 
        return self.classifier(fused).squeeze()

model = TopologyGatedMLP().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
MLP_EPOCHS = 15

print("\nTraining TG-MLP Architecture...")
for epoch in range(1, MLP_EPOCHS + 1):
    model.train()
    total_loss = 0
    
    for batch_n2v, batch_heur, batch_y in train_loader:
        batch_n2v, batch_heur, batch_y = batch_n2v.to(DEVICE), batch_heur.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch_n2v, batch_heur)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    model.eval()
    with torch.no_grad():
        pos_logits = model(X_val_pos_n2v.to(DEVICE), X_val_pos_heur.to(DEVICE))
        neg_logits = model(X_val_neg_n2v.to(DEVICE), X_val_neg_heur.to(DEVICE))
        pos_preds, neg_preds = torch.sigmoid(pos_logits).cpu(), torch.sigmoid(neg_logits).cpu()
        res = evaluator.eval({'y_pred_pos': pos_preds, 'y_pred_neg': neg_preds})
        
    print(f"Epoch {epoch:02d}/{MLP_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Val Hits@50: {res['hits@50']:.4%}")

print("\nTG-MLP Training Complete.")

--- Initializing Topology-Gated PyTorch MLP ---

Training TG-MLP Architecture...
Epoch 01/15 | Loss: 0.0131 | Val Hits@50: 61.7852%
Epoch 02/15 | Loss: 0.0099 | Val Hits@50: 62.0015%
Epoch 03/15 | Loss: 0.0096 | Val Hits@50: 62.0681%
Epoch 04/15 | Loss: 0.0095 | Val Hits@50: 61.1860%
Epoch 05/15 | Loss: 0.0094 | Val Hits@50: 62.2146%
Epoch 06/15 | Loss: 0.0092 | Val Hits@50: 60.9613%
Epoch 07/15 | Loss: 0.0093 | Val Hits@50: 61.0695%
Epoch 08/15 | Loss: 0.0091 | Val Hits@50: 62.5508%
